# Predicción de tiempos de entrega en streaming

Notebook operativo ejecutado por Papermill desde `predict_stream.sh`. Conserva los topics, UUID y destinos MongoDB/Kafka/Elasticsearch del flujo Docker.


In [ ]:
import json, math, time, requests, pymongo
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, abs as sql_abs, radians, sin, cos, asin, sqrt, dayofweek, hour, lit, to_json, struct
from pyspark.sql.types import StructType,StructField,StringType,TimestampType,DoubleType,LongType
from pyspark.ml import PipelineModel

spark=(SparkSession.builder.appName('Predict-Food-Delivery-Time-Streaming')
 .master('spark://agile:7077')
 .config('spark.jars.packages','org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.6').getOrCreate())
spark.sparkContext.setLogLevel('WARN')
model=PipelineModel.load('/home/jovyan/Food_delivery/models/pipeline_model.bin')


In [ ]:
schema=StructType([
 StructField('UUID',StringType(),False),StructField('delivery_person_age',DoubleType(),True),
 StructField('delivery_person_ratings',DoubleType(),True),StructField('restaurant_latitude',DoubleType(),True),
 StructField('restaurant_longitude',DoubleType(),True),StructField('delivery_location_latitude',DoubleType(),True),
 StructField('delivery_location_longitude',DoubleType(),True),StructField('order_date_and_time',TimestampType(),True),
 StructField('weather_conditions',StringType(),True),StructField('road_traffic_density',StringType(),True),
 StructField('vehicle_condition',DoubleType(),True),StructField('type_of_order',StringType(),True),
 StructField('type_of_vehicle',StringType(),True),StructField('multiple_deliveries',DoubleType(),True),
 StructField('festival',StringType(),True),StructField('city',StringType(),True)
])
raw=(spark.readStream.format('kafka').option('kafka.bootstrap.servers','kafka:9092')
 .option('subscribe','mydata_prediction_request').option('startingOffsets','latest').load())
df=raw.selectExpr('CAST(value AS STRING) json_data').select(from_json('json_data',schema).alias('data')).select('data.*')


In [ ]:
lat1=radians(sql_abs(col('restaurant_latitude'))); lon1=radians(sql_abs(col('restaurant_longitude')))
lat2=radians(sql_abs(col('delivery_location_latitude'))); lon2=radians(sql_abs(col('delivery_location_longitude')))
dlat=lat2-lat1; dlon=lon2-lon1
a=sin(dlat/2)*sin(dlat/2)+cos(lat1)*cos(lat2)*sin(dlon/2)*sin(dlon/2)
enriched=(df
 .withColumn('restaurant_latitude',sql_abs(col('restaurant_latitude')))
 .withColumn('restaurant_longitude',sql_abs(col('restaurant_longitude')))
 .withColumn('delivery_location_latitude',sql_abs(col('delivery_location_latitude')))
 .withColumn('delivery_location_longitude',sql_abs(col('delivery_location_longitude')))
 .withColumn('distance_km',lit(2*6371.0)*asin(sqrt(a)))
 .withColumn('day_of_week',dayofweek('order_date_and_time').cast('double'))
 .withColumn('hour_sin',sin(lit(2*math.pi)*hour('order_date_and_time')/lit(24.0)))
 .withColumn('hour_cos',cos(lit(2*math.pi)*hour('order_date_and_time')/lit(24.0))))
pred=model.transform(enriched)
result=pred.select('UUID','prediction','order_date_and_time','distance_km','road_traffic_density','weather_conditions','city')


In [ ]:
def write_outputs(batch_df,epoch_id):
    (batch_df.withColumn('key',col('UUID').cast('string'))
     .withColumn('value',to_json(struct('UUID','prediction','distance_km')))
     .selectExpr('CAST(key AS STRING)','CAST(value AS STRING)')
     .write.format('kafka').option('kafka.bootstrap.servers','kafka:9092')
     .option('topic','mydata_prediction_response').save())
    docs=[r.asDict(recursive=True) for r in batch_df.collect()]
    if not docs: return
    for d in docs:
        for k,v in list(d.items()):
            if hasattr(v,'isoformat'): d[k]=v.isoformat()
        d['prediction']=float(d['prediction']); d['distance_km']=float(d['distance_km']); d['epoch_id']=int(epoch_id)
    client=pymongo.MongoClient('mongo',27017)
    collection=client['agile_data_science']['mydata_prediction_response']
    for d in docs: collection.replace_one({'UUID':d['UUID']},d,upsert=True)
    client.close()
    payload=''.join(json.dumps({'index':{'_index':'mydata_prediction_response','_id':d['UUID']}})+'\n'+json.dumps(d)+'\n' for d in docs)
    requests.post('http://elastic:9200/_bulk',data=payload.encode(),headers={'Content-Type':'application/x-ndjson'},timeout=15).raise_for_status()

query=(result.writeStream.outputMode('append').foreachBatch(write_outputs)
 .option('checkpointLocation','/home/jovyan/logs/checkpoints/prediction-v2').start())
print('STREAMING_READY model=food-delivery-time-v2 topic=mydata_prediction_request')
query.awaitTermination()
